# Audio Feature Extraction Pipeline — Full Walkthrough

**Script:** `extract_features_folder.py`  
**Author Note:** This notebook documents the complete logic, data flow, and output structure of the feature extraction script used to process industrial audio recordings for anomaly detection.

---

## Table of Contents

1. [Overview & Goal](#1-overview--goal)
2. [Directory Architecture](#2-directory-architecture)
3. [Code Walkthrough](#3-code-walkthrough)
   - 3.1 [Imports & Dependencies](#31-imports--dependencies)
   - 3.2 [The `extract_features` Function](#32-the-extract_features-function)
   - 3.3 [Feature Reference Table](#33-feature-reference-table)
   - 3.4 [Directory Traversal Loop](#34-directory-traversal-loop)
4. [Output Pipeline](#4-output-pipeline)
5. [Execution Demo](#5-execution-demo)

---

## 1. Overview & Goal

The script `extract_features_folder.py` is a **batch audio feature extractor** designed for the MIMII (Malfunctioning Industrial Machine Investigation and Inspection) style dataset. Its job is to:

1. **Traverse** a structured directory tree of `.wav` audio recordings from industrial components (e.g., pumps, valves, bearings).
2. **Process** each audio file using [librosa](https://librosa.org/), a Python library for audio and music analysis.
3. **Extract** a fixed-size vector of acoustic features from each audio clip — capturing tonal, spectral, and energy-based characteristics of the sound.
4. **Aggregate** all extracted feature vectors into a tabular structure (a `pandas.DataFrame`) and **save** one CSV file per `(component_type, component_id, state)` combination.

The end goal is to convert raw `.wav` files into **machine-learning-ready feature matrices** where each row represents one audio file and each column is one acoustic feature. These CSVs can then be fed directly into classifiers (e.g., Isolation Forest, SVM, Autoencoder) to distinguish normal from abnormal machine operation.

---

## 2. Directory Architecture

The script assumes a specific nested directory layout. Understanding this structure is critical before running the script.

```
components/
│
├── pump/                          ← component_type
│   ├── id_00/                     ← component_type_id
│   │   ├── normal/                ← state
│   │   │   ├── 00000000.wav
│   │   │   ├── 00000001.wav
│   │   │   └── ...
│   │   └── abnormal/              ← state
│   │       ├── 00000000.wav
│   │       └── ...
│   └── id_02/
│       ├── normal/
│       └── abnormal/
│
├── valve/                         ← component_type
│   ├── id_00/
│   │   ├── normal/
│   │   └── abnormal/
│   └── id_04/
│       ├── normal/
│       └── abnormal/
│
└── bearing/                       ← component_type
    ├── id_00/
    │   ├── normal/
    │   └── abnormal/
    └── id_02/
        ├── normal/
        └── abnormal/
```

**Key observations:**
- The tree has exactly **4 levels of depth** below the root `components/` folder.
- Level 1 — **component type**: the machine category (pump, valve, bearing, fan, slider, etc.)
- Level 2 — **component ID**: a specific physical unit of that machine type (id_00, id_02, id_04, …)
- Level 3 — **state**: either `normal` (healthy operation) or `abnormal` (faulty operation)
- Level 4 — **audio files**: the individual `.wav` recordings for that machine/state combination

---

## 3. Code Walkthrough

### 3.1 Imports & Dependencies

The script relies on four libraries:

In [ ]:
import librosa   # Audio loading and feature extraction
import numpy as np   # Numerical operations (mean, std)
import pandas as pd  # Tabular data structures and CSV I/O
import os           # Filesystem traversal

| Library | Role in this script |
|---------|--------------------|
| `librosa` | Loads `.wav` files and computes all acoustic features (MFCCs, spectral descriptors, RMS, ZCR) |
| `numpy` | Computes per-feature summary statistics: `mean` and `std` across time frames |
| `pandas` | Builds per-file feature rows, concatenates them, and writes the final CSV |
| `os` | Iterates over directory contents at each level of the nested folder hierarchy |

### 3.2 The `extract_features` Function

This is the heart of the pipeline. It takes a single `.wav` file path, loads the audio, computes seven categories of acoustic features, and returns them as a flat Python dictionary.

**Signature:**
```python
def extract_features(wav_path, sr=22050) -> dict
```

**Parameters:**
- `wav_path` — Absolute path to the `.wav` audio file.
- `sr=22050` — Target sample rate in Hz. `librosa.load` resamples the audio to this rate if needed. 22050 Hz is the standard for audio ML tasks.

---

**Step 1 — Load audio**

In [ ]:
# Step 1: Load the .wav file as a 1-D numpy array 'y' (mono waveform)
# librosa.load resamples to `sr` Hz and converts to float32
y, sr = librosa.load(wav_path, sr=sr)

- `y` is a 1-D NumPy array of audio samples (amplitude values over time).
- `sr` is the actual sample rate after loading (will equal 22050 due to the parameter).
- The audio is always loaded as **mono** (librosa default). Stereo files are mixed down automatically.

---

**Step 2 — Extract MFCCs (Mel-Frequency Cepstral Coefficients)**

In [ ]:
# Step 2: Compute 13 MFCC coefficients
# Output shape: (13, T) — 13 coefficients x T time frames
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

for i in range(13):
    features[f"mfcc_{i+1}_mean"] = np.mean(mfccs[i])  # Average over time
    features[f"mfcc_{i+1}_std"]  = np.std(mfccs[i])   # Variability over time

**What are MFCCs?**  
MFCCs represent the **short-term power spectrum of sound on a mel (perceptual) frequency scale**. Each coefficient captures energy in a different frequency band. They are the single most important feature family for audio classification — they encode *timbre* (the tonal "texture" of the sound).

- `n_mfcc=13` computes 13 coefficients. MFCC-1 captures overall energy; MFCC-2 through MFCC-13 capture spectral shape at increasingly fine resolution.
- Because the output is a 2-D matrix `(13, T)`, the script summarises each coefficient over time using `mean` (average level) and `std` (how much it fluctuates).
- This produces **26 features**: `mfcc_1_mean … mfcc_13_mean` and `mfcc_1_std … mfcc_13_std`.

---

**Step 3 — Spectral Centroid**

In [ ]:
sc = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
features["spectral_centroid_mean"] = np.mean(sc)
features["spectral_centroid_std"]  = np.std(sc)

The spectral centroid is the **"centre of mass" of the frequency spectrum** — the frequency around which most of the signal's energy is concentrated. A high centroid means the sound is dominated by high frequencies (e.g., a screeching valve); a low centroid means low-frequency dominance (e.g., a deep bearing rumble).

---

**Step 4 — Spectral Bandwidth**

In [ ]:
sb = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
features["spectral_bandwidth_mean"] = np.mean(sb)
features["spectral_bandwidth_std"]  = np.std(sb)

Spectral bandwidth measures the **width of the frequency band** around the centroid. It captures how "spread out" or "focused" the frequency content is. A malfunctioning machine often exhibits broadband noise (high bandwidth) vs. tonal, narrow-band sounds under normal operation.

---

**Step 5 — Spectral Flatness**

In [ ]:
sf = librosa.feature.spectral_flatness(y=y)[0]
features["spectral_flatness_mean"] = np.mean(sf)
features["spectral_flatness_std"]  = np.std(sf)

Spectral flatness (also called Wiener entropy) quantifies how **noise-like vs. tone-like** a sound is.  
- Value near **1.0** → white noise (energy spread evenly across all frequencies)
- Value near **0.0** → pure tone (energy concentrated at a single frequency)

Abnormal machines often generate more irregular, noisy sounds that show higher flatness values.

---

**Step 6 — Spectral Rolloff**

In [ ]:
sr_feat = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
features["spectral_rolloff_mean"] = np.mean(sr_feat)

Spectral rolloff is the **frequency below which 85% (librosa default) of the spectral energy is concentrated**. It gives a sense of the upper frequency boundary of the signal and is useful for distinguishing sounds with different high-frequency content.

> **Note:** Only the `mean` is stored here — unlike the other spectral features, no `std` is recorded for rolloff.

---

**Step 7 — RMS Energy**

In [ ]:
rms = librosa.feature.rms(y=y)[0]
features["rms_energy_mean"] = np.mean(rms)
features["rms_energy_std"]  = np.std(rms)

RMS (Root Mean Square) energy measures the **loudness or power of the audio signal** over time. It reflects the overall amplitude envelope. Machines with unusual vibrations or grinding sounds often have elevated or more variable RMS energy.

---

**Step 8 — Zero-Crossing Rate (ZCR)**

In [ ]:
zcr = librosa.feature.zero_crossing_rate(y)[0]
features["zcr_mean"] = np.mean(zcr)
features["zcr_std"]  = np.std(zcr)

ZCR counts **how often the audio waveform crosses the zero amplitude line** per unit time. High ZCR typically indicates a noisy, high-frequency, or percussive sound. It is a computationally cheap feature that still carries useful discriminative signal.

---

### 3.3 Feature Reference Table

The function returns a flat dictionary with **37 keys** in total:

| Feature Group | Feature Names | Count | Description |
|---------------|--------------|-------|-------------|
| MFCCs | `mfcc_1_mean` … `mfcc_13_mean` | 13 | Mean of each MFCC coefficient over all time frames |
| MFCCs | `mfcc_1_std` … `mfcc_13_std` | 13 | Std dev of each MFCC coefficient over all time frames |
| Spectral Centroid | `spectral_centroid_mean`, `spectral_centroid_std` | 2 | Centre of spectral mass |
| Spectral Bandwidth | `spectral_bandwidth_mean`, `spectral_bandwidth_std` | 2 | Width of the spectral band |
| Spectral Flatness | `spectral_flatness_mean`, `spectral_flatness_std` | 2 | Noise-like vs tone-like ratio |
| Spectral Rolloff | `spectral_rolloff_mean` | 1 | Frequency containing 85% of energy |
| RMS Energy | `rms_energy_mean`, `rms_energy_std` | 2 | Signal loudness/power |
| Zero-Crossing Rate | `zcr_mean`, `zcr_std` | 2 | Rate of sign changes in the waveform |
| **Total** | | **37** | |

---

### 3.4 Directory Traversal Loop

After the function definition, the script runs a 4-level nested `for` loop that drives the entire batch processing pipeline:

In [ ]:
# Shown here with corrected paths for illustration
BASE = r"C:\Users\sanid\PycharmProjects\Listen"
COMPONENTS_DIR = os.path.join(BASE, "components")

df = pd.DataFrame()  # Global accumulator — reset between saves below

for comp_type in os.listdir(COMPONENTS_DIR):                              # Level 1: pump, valve, bearing
    for comp_type_id in os.listdir(f"{COMPONENTS_DIR}\\{comp_type}"):    # Level 2: id_00, id_02
        for comp_type_id_state in os.listdir(                             # Level 3: normal, abnormal
                f"{COMPONENTS_DIR}\\{comp_type}\\{comp_type_id}"):
            for filename in os.listdir(                                   # Level 4: *.wav files
                    f"{COMPONENTS_DIR}\\{comp_type}\\{comp_type_id}\\{comp_type_id_state}"):

                # Build full path and extract features for one file
                wav_path = f"{COMPONENTS_DIR}\\{comp_type}\\{comp_type_id}\\{comp_type_id_state}\\{filename}"
                features = extract_features(wav_path)

                # Append this file's feature row to the running DataFrame
                df_features = pd.DataFrame([features])
                df = pd.concat([df, df_features], ignore_index=True)
                print("Done with ", filename)

            print(f"Done with {comp_type_id}")

            # Save CSV after processing all files in a (comp_type, id, state) group
            df.to_csv(f"{comp_type}_{comp_type_id}_{comp_type_id_state}.csv", index=False)

**Loop-by-loop breakdown:**

| Loop variable | What it iterates | Example values |
|---------------|-----------------|----------------|
| `comp_type` | All subdirectories inside `components/` | `pump`, `valve`, `bearing` |
| `comp_type_id` | All subdirectories inside `components/<comp_type>/` | `id_00`, `id_02`, `id_04` |
| `comp_type_id_state` | All subdirectories inside `components/<comp_type>/<id>/` | `normal`, `abnormal` |
| `filename` | All files inside the state folder | `00000000.wav`, `00000001.wav` |

**Data flow inside the innermost loop:**
1. `extract_features(wav_path)` → returns a 37-key dictionary.
2. `pd.DataFrame([features])` → wraps it into a single-row DataFrame.
3. `pd.concat([df, df_features])` → appends the new row to the running accumulator.

**After the 3rd-level loop completes** (all files for one `comp_type / id / state`), `df.to_csv(...)` writes the accumulated rows to disk.

> **Implementation Note:** The global `df` is not reset between iterations of the outer loops. This means each CSV will contain the rows accumulated across all previously processed states and IDs in addition to its own — a bug in the original script. In a corrected version, `df = pd.DataFrame()` should be placed just before the innermost loop (or after the save).

---

## 4. Output Pipeline

### What gets saved and where

At the end of each state-level iteration, the script calls:
```python
df.to_csv(f"{comp_type}_{comp_type_id}_{comp_type_id_state}.csv", index=False)
```

This produces one CSV file per `(component_type, component_id, state)` triple in the **current working directory**.

**Expected output files:**
```
pump_id_00_normal.csv
pump_id_00_abnormal.csv
pump_id_02_normal.csv
pump_id_02_abnormal.csv
valve_id_00_normal.csv
valve_id_00_abnormal.csv
...
```

### CSV column structure

Each CSV has exactly **37 columns** — one per extracted feature — and **N rows** where N equals the number of `.wav` files in that state folder. Each row represents one audio recording.

In [ ]:

import librosa
import numpy as np
import pandas as pd

dummy_features = {}

for i in range(1, 14):
    dummy_features[f"mfcc_{i}_mean"] = 0.0
    dummy_features[f"mfcc_{i}_std"]  = 0.0

for name in ["spectral_centroid", "spectral_bandwidth", "spectral_flatness", "rms_energy", "zcr"]:
    dummy_features[f"{name}_mean"] = 0.0
    dummy_features[f"{name}_std"]  = 0.0

dummy_features["spectral_rolloff_mean"] = 0.0

df_schema = pd.DataFrame([dummy_features])
print(f"Total columns: {len(df_schema.columns)}")
print("\nAll column names:")
for col in df_schema.columns:
    print(" ", col)

### Visual summary of the output pipeline

```
components/
├── pump/
│   ├── id_00/
│   │   ├── normal/     [N .wav files]  ──→  pump_id_00_normal.csv   (N rows × 37 cols)
│   │   └── abnormal/   [M .wav files]  ──→  pump_id_00_abnormal.csv (M rows × 37 cols)
│   └── id_02/
│       ├── normal/     [N .wav files]  ──→  pump_id_02_normal.csv
│       └── abnormal/   [M .wav files]  ──→  pump_id_02_abnormal.csv
└── valve/
    └── id_00/
        ├── normal/     [N .wav files]  ──→  valve_id_00_normal.csv
        └── abnormal/   [M .wav files]  ──→  valve_id_00_abnormal.csv
```

---

## 5. Execution Demo

### Option A — Run the script directly from the notebook shell

In [ ]:
# Run the script from the notebook (adjust path as needed)
import subprocess
result = subprocess.run(
    ["python", "extract_features_folder.py"],
    capture_output=True,
    text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

### Option B — Call `extract_features` on a single file inline

In [ ]:
import sys
import os

# Add the script's directory to the path so we can import from it
SCRIPT_DIR = r"C:\Users\sanid\PycharmProjects\Listen\Listen_repo\feature_extractor"
sys.path.insert(0, SCRIPT_DIR)

from extract_features_folder import extract_features

# Point to a single sample audio file
sample_wav = r"C:\Users\sanid\PycharmProjects\Listen\components\pump\id_00\normal\00000000.wav"

if os.path.exists(sample_wav):
    features = extract_features(sample_wav)
    result_df = pd.DataFrame([features])
    print(f"Extracted {len(result_df.columns)} features from: {os.path.basename(sample_wav)}")
    display(result_df.T.rename(columns={0: "value"}))
else:
    print(f"File not found: {sample_wav}")
    print("Update 'sample_wav' to a valid .wav path and re-run.")

### Option C — Process a single component type with a corrected, self-contained loop

In [ ]:
import librosa
import numpy as np
import pandas as pd
import os
from extract_features_folder import extract_features

BASE_DIR = r"C:\Users\sanid\PycharmProjects\Listen\components"
TARGET_COMP = "pump"  # Change to 'valve', 'bearing', etc.

comp_dir = os.path.join(BASE_DIR, TARGET_COMP)

for comp_id in os.listdir(comp_dir):
    id_dir = os.path.join(comp_dir, comp_id)
    for state in os.listdir(id_dir):
        state_dir = os.path.join(id_dir, state)

        rows = []  # Fresh list per (comp, id, state) — avoids accumulation bug
        for filename in os.listdir(state_dir):
            if not filename.endswith(".wav"):
                continue
            wav_path = os.path.join(state_dir, filename)
            features = extract_features(wav_path)
            rows.append(features)
            print(f"  Processed: {filename}")

        df_out = pd.DataFrame(rows)
        out_filename = f"{TARGET_COMP}_{comp_id}_{state}.csv"
        df_out.to_csv(out_filename, index=False)
        print(f"Saved {len(df_out)} rows → {out_filename}")

### Verify the output CSV

In [ ]:
# Load and inspect one of the generated CSV files
csv_to_check = "pump_id_00_normal.csv"

if os.path.exists(csv_to_check):
    df_check = pd.read_csv(csv_to_check)
    print(f"Shape: {df_check.shape}  ({df_check.shape[0]} audio files × {df_check.shape[1]} features)")
    print("\nFirst 3 rows (transposed for readability):")
    display(df_check.head(3).T)
    print("\nBasic statistics:")
    display(df_check.describe())
else:
    print(f"{csv_to_check} not found — run the extraction cells above first.")

---

## Summary

| Stage | What happens |
|-------|--------------|
| **Input** | Nested `components/<type>/<id>/<state>/*.wav` directory tree |
| **Per-file processing** | `librosa.load` → 7 feature families → 37-value dictionary |
| **Aggregation** | `pd.concat` builds a growing DataFrame of all file rows |
| **Output** | One `<type>_<id>_<state>.csv` per leaf group, 37 columns, N rows |
| **Downstream use** | CSVs are fed into anomaly detection / classification models |

The 37 features span **spectral** (MFCCs, centroid, bandwidth, flatness, rolloff), **energy** (RMS), and **temporal** (ZCR) domains — together providing a rich, compact fingerprint of each audio recording suitable for distinguishing normal from abnormal machine operation.